# 4. Découverte des LLM avec Ollama

**Un LLM (*Large Language Model*)** a appris, sur d'énormes quantités de texte, à **prédire la suite la plus probable d'un texte**.
Il ne consulte aucune base de données : il génère du texte *plausible*, pas forcément *vrai*.

**Ollama** fait tourner ces modèles en local, sur mon PC : le code Python l'interroge comme un petit serveur.
Ce notebook explore trois notions : la conversation par messages, la température, et la limite fondamentale des LLM.

## Un premier échange
Une conversation est une **liste de messages**, chacun avec un **rôle** (`user` = l'utilisateur) et un **contenu**.
Le texte de la réponse se trouve dans `reponse["message"]["content"]`.

In [1]:
import ollama

reponse = ollama.chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": "Explique en une phrase ce qu'est un accident corporel de la route."}],
)
print(reponse["message"]["content"])

Un accident corporel de la route est un incident sur les routes où des personnes sont blessées ou meurent à cause de conditions imprévues, conduites imprudentes ou d'autres facteurs liés au trafic.


## Le prompt système et la température
- Le message `system` fixe les **règles du jeu** pour toute la conversation : personnage, langue, longueur.
- La **température** règle la part de hasard. À chaque mot, le modèle calcule une probabilité pour chaque suite possible : à 0, il choisit toujours la plus probable ; à 1, il tire au sort selon ces probabilités.

Le modèle n'a **aucune mémoire** : pour poursuivre une conversation, on lui renvoie tout l'historique à chaque appel.

In [2]:
messages = [
    # Le prompt SYSTÈME fixe le comportement du modèle pour toute la conversation
    {"role": "system", "content": "Tu es un analyste en sécurité routière. Tu réponds en français, en 2 phrases maximum."},
    {"role": "user", "content": "Pourquoi les accidents sont-ils plus graves la nuit ?"},
]

# La température règle la part de hasard : 0 = toujours la réponse la plus probable, 1 = plus varié
for temperature in [0, 0, 1.0, 1.0]:
    r = ollama.chat(model="qwen2.5:3b", messages=messages, options={"temperature": temperature})
    print(f"--- température {temperature}\n{r['message']['content']}\n")

--- température 0
Les routes sont moins visibles et la vision est limitée, augmentant ainsi le risque d'accidents. La réduction de la vitesse due à la pénombre augmente également les risques en cas d'urgence.

--- température 0
Les routes sont moins visibles et les conducteurs ont moins de réflexion due à l'obscurité, augmentant ainsi le risque d'accidents. Les vitres chauffées peuvent également favoriser les collisions avec des objets froids.

--- température 1.0
Les faibles conditions de vision nuit et la vision ophtalmologique réduite incitent à des comportements plus imprudents. La circulation est aussi plus dense, augmentant le risque d'accident.

--- température 1.0
Les routes sont moins fréquentées la nuit, augmentant l'impact potentiel des accidents. La vision limitée et l'accoutumance aux ombres rendent le détection des dangers plus difficile.



**Observations :**
- À température 0, les deux réponses sont proches mais pas identiques. La seconde parle de « vitres chauffées » qui favoriseraient les collisions : une phrase fluide mais dénuée de sens, c'est une **hallucination**. Pour des réponses parfaitement reproductibles, il faut aussi fixer une graine (`seed`).
- À température 1, les réponses se **contredisent** : l'une dit que la circulation est plus dense la nuit, l'autre (comme les données) qu'elle l'est moins.

→ Règle retenue pour la suite : **température 0 pour toute tâche factuelle** (chiffres, SQL).

## La limite fondamentale : le modèle ne connaît pas mes données

In [3]:
r = ollama.chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": "Combien de personnes ont été tuées sur les routes en France en 2024 ? Donne un nombre précis."}],
    options={"temperature": 0},
)
print(r["message"]["content"])

Je suis désolé pour la confusion, mais je n'ai pas accès aux données spécifiques concernant le nombre exact de personnes tuées sur les routes en France pour l'année 2024. Les statistiques des accidents routiers sont mises à jour régulièrement et je ne peux pas fournir d'informations sur une année précise sans vérification auprès d'une source fiable. Je vous recommande de consulter le site officiel du ministère français de la sécurité intérieure ou un service similaire pour obtenir cette information exacte.


Le modèle a été entraîné avant 2024 et n'a pas accès à internet : il ne peut pas connaître le chiffre (3 432 tués selon mon analyse).
Ici, il reconnaît ne pas savoir, mais souvent un modèle **invente** un nombre plausible.

On ne peut pas lui « donner toute la base » : 125 000 lignes ne tiennent pas dans un message, et un LLM calcule mal.
D'où le principe qui guide toute la suite du projet : **le code calcule, le LLM rédige.**
- Notebook 05 : le code calcule les chiffres en SQL, le LLM les met en forme.
- Notebook 06 : le LLM écrit la requête SQL, la base calcule la réponse.